<a href="https://colab.research.google.com/github/I-yuki-0424/Decision-Process-order-driven/blob/main/docs/JP-ideas/DPOD.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Decision Process Order-Driven

## To best decision system

## Name and Mean

|                  |main | 2nd |
| ---------------- | --- | --- |
| action           | A   |     |
| State            | S   | S_1 |
| target           | T   |     |
| history          | H   |     |
| compression unit | Z   |     |
| candidates       | K   |     |


In [ ]:
# just import
import jax
import jax.numpy as jnp
import flax.linen as nn

In [ ]:
def get_sinusoidal_positional_encoding(seq_len, d_model):
    """
    sin cos encoder
    """
    position = jnp.arange(seq_len)[:, None]
    div_term = jnp.exp(jnp.arange(0, d_model, 2) * (-jnp.log(10000.0) / d_model))

    pe_sin = jnp.sin(position * div_term)
    pe_cos = jnp.cos(position * div_term)

    pe = jnp.zeros((seq_len, d_model))
    pe = pe.at[:, 0::2].set(pe_sin)
    pe = pe.at[:, 1::2].set(pe_cos)
    return pe

In [ ]:
def _y_mdp_action_single(array_to_action, k):
    """
    Original per-sample body of y_mdp_action, unmodified. Operates on a
    single (N, b) slice. Kept separate from y_mdp_action so other batched
    functions can call the per-sample logic directly without a nested vmap.
    """
    S    = array_to_action[:, 0].astype(jnp.int32)
    S_1  = array_to_action[:, 1].astype(jnp.int32)
    A    = array_to_action[:, 2].astype(jnp.int32)
    P    = array_to_action[:, 3]
    R    = array_to_action[:, 4]
    G    = array_to_action[:, 5]

    n = array_to_action.shape[0]                 # static upper bound on state/action ids
    state_action_id = S * n + A                   # unique id per (s, a), valid since A < n

    Q0 = jnp.zeros((n, n), dtype=array_to_action.dtype)  # Q*(s, a) initialized to 0

    def cond_fn(carry):
        it, Q, delta = carry
        return jnp.logical_and(it < n, delta > 1e-6)

    def body_fn(carry):
        it, Q, _ = carry
        topk_next = jax.lax.top_k(Q[S_1], k)[0]         # (num_rows, k)
        soft_max_next = jnp.mean(topk_next, axis=-1)     # (num_rows,)
        transition_value = P * (R + G * soft_max_next)   # (num_rows,)
        Q_flat = jax.ops.segment_sum(transition_value, state_action_id, num_segments=n * n)
        Q_new = Q_flat.reshape(n, n)
        delta = jnp.max(jnp.abs(Q_new - Q))
        return (it + 1, Q_new, delta)

    init_carry = (0, Q0, jnp.asarray(jnp.inf, dtype=array_to_action.dtype))
    _, Q, _ = jax.lax.while_loop(cond_fn, body_fn, init_carry)

    top_values, top_actions = jax.lax.top_k(Q, k)

    return top_actions, top_values


@jax.jit
def y_mdp_action(array_to_action, k):
    """
    This is simple **Bellman optimality equation**
    Latex
    Q^*(s, a) = \\sum_{s'} P(s' | s, a) \\left[ R(s, a, s') + \\gamma \\max_{a'} Q^*(s', a') \\right]

    Batch dimension added: array_to_action now has shape (B, N, b) -- B
    batches, N state-action rows per batch, b=6 feature columns (S, S_1,
    A, P, R, G). Per-sample logic is unchanged; jax.vmap maps
    `_y_mdp_action_single` over the leading batch axis. k is shared across
    the batch (unbatched), matching its original role as a fixed top-k
    width.
    """
    return jax.vmap(_y_mdp_action_single, in_axes=(0, None))(array_to_action, k)

In [ ]:
def _Y_attention_state_5_1_Bace_single(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k
):
    """Original per-sample body, unmodified. Operates on (N, b) slices."""
    # 1. actions, goal, state concatenate
    q_ags_init = jnp.concatenate([actions, target, state, histry], axis=0)

    def layer_fn(q_carry, _):
        y_attn = nn.dot_product_attention(
            query=q_carry,
            key=k_s,
            value=v_s,
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        # Residual connection style update
        return q_carry + y_attn, None

    # 2. Layering according to num_l
    y_attention_state, _ = jax.lax.scan(layer_fn, q_ags_init, jnp.arange(num_l))

    mdp_actions, mdp_target, mdp_state, mdp_histry = nn.split(y_attention_state[actions.shape[0],
                                                            (actions.shape[0] + target.shape[0]),
                                                            (actions.shape[0] + target.shape[0] + state.shape[0])], axis= 0)
    mdp_array = jnp.column_stack([state, mdp_state, actions, mdp_actions, (target - mdp_state), (target - mdp_state)* 1/num_step])
    Y_action = _y_mdp_action_single(mdp_array, k)

    return Y_action


@jax.jit
def Y_attention_state_5_1_Bace(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k
):
    """Simple and 5th idea bace

    Batch dimension added: actions, target, state, histry, k_s, v_s,
    mask_s now have shape (B, N, b). dropout_rate_s, dropout_enabled_s,
    rng_s, num_l, num_step, k are treated as shared across the batch
    (unbatched hyperparameters/config). The internal call to y_mdp_action
    uses its single-sample form (`_y_mdp_action_single`) to avoid a nested
    vmap, since this function is itself vmapped over the batch axis.
    """
    return jax.vmap(
        _Y_attention_state_5_1_Bace_single,
        in_axes=(0, 0, 0, 0, 0, 0, None, None, None, 0, None, None, None)
    )(actions, target, state, histry, k_s, v_s, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, num_step, k)

In [ ]:
def _Y_attention_state_5_2_single(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s,
    w_dense, b_dense, num_l,
    mask_histry=None
):
    """Original per-sample body, unmodified. Operates on (N, b) slices."""
    # Prepare initial inputs
    q_ags_init = jnp.concatenate([actions, target, state], axis=0)
    seq_len, d_model = histry.shape[-2], histry.shape[-1]
    pe = get_sinusoidal_positional_encoding(seq_len, d_model)
    histry_pe_init = histry + pe

    def layer_fn(carry, _):
        q_ags, h_pe = carry

        # 1. actions, goal, state attention
        y_ags = nn.dot_product_attention(
            query=q_ags,
            key=k_s,
            value=v_s,
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )

        # 2. attention to history
        y_histry = nn.dot_product_attention(
            query=h_pe,
            key=h_pe,
            value=h_pe,
            bias=mask_histry,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )

        # 3. linear integration and residual update
        y_combined = jnp.concatenate([y_ags, y_histry], axis=0)
        out = jnp.dot(y_combined, w_dense) + b_dense

        # Splitting back to maintain shapes for next layer (approximate residual logic)
        # Note: In a real deep model, shapes must align for addition/layering
        return (out[:q_ags.shape[0]], out[q_ags.shape[0]:]), None

    (final_q, final_h), _ = jax.lax.scan(layer_fn, (q_ags_init, histry_pe_init), jnp.arange(num_l))

    # Final combined state
    y_attention_state = jnp.concatenate([final_q, final_h], axis=0)
    return y_attention_state


@jax.jit
def Y_attention_state_5_2(
    actions, target, state, histry,
    k_s, v_s,
    dropout_rate_s, dropout_enabled_s, rng_s, mask_s,
    w_dense, b_dense, num_l,
    mask_histry=None
):
    """
        idea5, type1 channel separation
        A function that separates the attention of actions,
        goals, and states from the self-attention of history, and integrates them using a fully connected layer.
        Layered according to num_l.

        Batch dimension added: actions, target, state, histry, k_s, v_s,
        mask_s, and mask_histry (when not None) now have shape (B, N, b).
        dropout_rate_s, dropout_enabled_s, rng_s, w_dense, b_dense, num_l
        are shared parameters/hyperparameters, unbatched.
    """
    return jax.vmap(
        _Y_attention_state_5_2_single,
        in_axes=(0, 0, 0, 0, 0, 0, None, None, None, 0, None, None, None, 0)
    )(actions, target, state, histry, k_s, v_s, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, w_dense, b_dense, num_l, mask_histry)

In [ ]:
def _Y_attention_state_5_3_L_1_single(actions, target, state, histry, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, mask_histry=None):
    """Original per-sample body, unmodified. Operates on (N, b) slices."""
    def make_qkv(actions, target, state, histry):
        q_seq_len = actions.shape[-2]
        q_d_model = actions.shape[-1]
        q_pe = get_sinusoidal_positional_encoding(q_seq_len, q_d_model)
        q = actions + q_pe

        k_bace = jnp.concatenate([target, state], axis=0)
        k_seq_len = k_bace.shape[-2]
        k_d_model = k_bace.shape[-1]
        k_pe = get_sinusoidal_positional_encoding(k_seq_len, k_d_model)
        k = k_bace + k_pe

        v_seq_len = histry.shape[-2]
        v_d_model = histry.shape[-1]
        v_pe = get_sinusoidal_positional_encoding(v_seq_len, v_d_model)
        v = histry + v_pe

        qkv = jnp.concatenate([q, k, v], axis=0)
        return (qkv, q_seq_len, k_seq_len, v_seq_len)


    def attention(carry, _):
        qkv = qkv_init + carry
        q, k, v = jnp.split(qkv, [q_split_idx, k_split_idx], axis=0)

        y_attention_state_carry_q = nn.dot_product_attention(
            query=q,
            key=q,
            value=q,
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_attention_state_carry_k = nn.dot_product_attention(

            query=k,
            key=k,
            value=k,
            bias=mask_s,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )
        y_attention_state_carry_v = nn.dot_product_attention(
            query=v,
            key=v,
            value=v,
            bias=mask_histry,
            dropout_rate=dropout_rate_s,
            deterministic=not dropout_enabled_s,
            dropout_rng=rng_s
        )

        y_attention_state_carry = jnp.concatenate([y_attention_state_carry_q, y_attention_state_carry_k, y_attention_state_carry_v], axis=0)

        return y_attention_state_carry, None

    qkv_init, q_seq_len, k_seq_len, v_seq_len = make_qkv(actions, target, state, histry)
    q_split_idx = q_seq_len
    k_split_idx = q_seq_len + k_seq_len

    carry_init = jnp.zeros_like(qkv_init)
    loop_trigger = jnp.arange(num_l)

    final_carry, _ = jax.lax.scan(attention, carry_init, loop_trigger)
    y_attention_state = qkv_init + final_carry

    return y_attention_state


@jax.jit
def Y_attention_state_5_3_L_1(actions, target, state, histry, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, mask_histry=None):
    """ idea5, type2 channel separation.
        channel is A T+S S
        Enable Position encoding

        Batch dimension added: actions, target, state, histry, mask_s, and
        mask_histry (when not None) now have shape (B, N, b).
        dropout_rate_s, dropout_enabled_s, rng_s, num_l are shared,
        unbatched.
    """
    return jax.vmap(
        _Y_attention_state_5_3_L_1_single,
        in_axes=(0, 0, 0, 0, None, None, None, 0, None, 0)
    )(actions, target, state, histry, dropout_rate_s, dropout_enabled_s, rng_s, mask_s, num_l, mask_histry)


In [ ]:
"""
The structure I am imagining has various flaws and deficiencies, and because I have only conveyed the concepts until now and left the coding entirely to others, the current situation is that problems are arising precisely because I am writing it this way to communicate the correct formulas.
First, I intended to incorporate self-attention into the design itself. This is because I want to assign relationships to how "now" relates to the "goal" and what actions can be taken.
However, I intend to insert an important W after that. I am giving physical constraints to W_K, and I expect that physically impossible candidates will become zero via the Softmax function.
My plan is to then have it predict the next goal, S'. To output S', I was thinking of compressing it into small arrays of B, 1, d in order using fully connected layers.
Then, for that state S', I intend to perform a search for actions using the MDP and the Bellman equation for the action-value function, using only the actions that remain after being filtered by the physical W_K, and have it output the top K candidates. Or, I am considering having it do the same thing using a Transformer. Finally, although I am not sure if it is in the documentation, I was thinking of compressing the k for each step, which are arranged by units called Z, into one per step and storing them in H."""

In [ ]:
def _Get_next_state_single(y_attention_state, state_index):
        """Original per-sample body, unmodified."""
        state_softmax = nn.softmax(y_attention_state[state_index])
        state_next = jnp.argmax(state_softmax)
        return (state_next)


@jax.jit
def Get_next_state(y_attention_state, state_index):
        """ This is Transformer thought best next state.

        Batch dimension added: y_attention_state now has shape (B, N, b).
        state_index is treated as shared across the batch (the same
        relative position is queried for every sample); pass a batched
        index array with in_axes changed to 0 if per-sample indices are
        needed instead.
        """
        return jax.vmap(_Get_next_state_single, in_axes=(0, None))(y_attention_state, state_index)

In [ ]:
def _make_array_to_action_single(state, next_state, actions, p_coe, r_coe, gamma_coe):
    """
    Original per-sample body. next_state is clamped away from zero before
    division to avoid inf/nan when a next_state entry is exactly (or very
    close to) zero. The ratio-based gradient formula itself is otherwise
    unchanged from the original.
    """
    eps = jnp.asarray(1e-8, dtype=next_state.dtype)
    sign = jnp.where(next_state >= 0, 1.0, -1.0).astype(next_state.dtype)
    safe_next_state = jnp.where(jnp.abs(next_state) < eps, sign * eps, next_state)
    gradient = state / safe_next_state * 100
    P = p_coe * gradient
    R = r_coe * gradient
    G = gamma_coe * gradient

    return (jnp.column_stack([state, next_state, actions, P, R, G]))


#sample
@jax.jit
def make_array_to_action(state, next_state, actions, p_coe, r_coe, gamma_coe):
    """
    S,S' = already,
    a = alredy,
    P = P(%)_coefficient * gradient of S -> S'
    R = R(%)_coefficient * gradient of S -> S'
    Gamma = G(%)_coefficient * gradient of S -> S'
    How to set coefficient? I dont know :/

    Batch dimension added: state, next_state, actions now have shape
    (B, N, b). p_coe, r_coe, gamma_coe are shared coefficients, unbatched.
    next_state is clamped away from zero in the per-sample body to avoid
    inf/nan (see _make_array_to_action_single).
    """
    return jax.vmap(
        _make_array_to_action_single, in_axes=(0, 0, 0, None, None, None)
    )(state, next_state, actions, p_coe, r_coe, gamma_coe)

In [ ]:
def _y_attention_action_single(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a):
    """Original per-sample body, unmodified."""
    q = array_to_action

    y_attention_action = nn.dot_product_attention(
        query = q,
        key = k_a,
        value = v_a,
        bias = mask_a,
        dropout_rate = dropout_rate_a,
        deterministic = not dropout_enabled_a,
        dropout_rng = rng_a
    )

    return (y_attention_action)


# Next, i think think next action to allive to state_next. and to cut cost, maybe MDP is better. but this is study so use transformer.
@jax.jit
def y_attention_action(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a):
    """This function return Next step **action**.
       but i think transformer is hevy. so just test
       array must include reword of action, % of go state when use actuion(i), gamma
       And, i think dropout must not use. It feels strange to train by logically dropping out groups of choices. I have no evidence for this.

       Batch dimension added: array_to_action, k_a, v_a, mask_a now have
       shape (B, N, b). dropout_rate_a, dropout_enabled_a, rng_a are
       shared, unbatched.
    """
    return jax.vmap(
        _y_attention_action_single, in_axes=(0, 0, 0, None, None, None, 0)
    )(array_to_action, k_a, v_a, dropout_rate_a, dropout_enabled_a, rng_a, mask_a)